# RNN

In [1]:
from keras.models import Sequential
from keras.layers import SimpleRNN,Dense
import numpy as np

vocab = ['d','e','p','<stop>']
char2idx = {c: i for i, c in enumerate(vocab)}
idx2char = {i: c for c, i in char2idx.items()}

seq = ['d','e','p']
target = ['e','p','<stop>']

X = np.eye(len(vocab))[[char2idx[c] for c in seq]].reshape(len(seq), 1, len(vocab))
y = np.eye(len(vocab))[[char2idx[c] for c in target]]

model = Sequential([SimpleRNN(8, input_shape=(1, len(vocab))),
                   Dense(len(vocab), activation='softmax')
                   ])

model.compile(loss='categorical_crossentropy',optimizer='adam')

model.fit(X,y,epochs=200,verbose=0)

test = np.eye(len(vocab))[[char2idx['p']]].reshape(1,1,len(vocab))

pred = model.predict(test)
print('Next char prediction :', idx2char[np.argmax(pred)])

2026-04-28 16:39:24.075444: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777394364.467299      22 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777394364.581792      22 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777394365.709808      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777394365.709846      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777394365.709849      22 computation_placer.cc:177] computation placer alr

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 344ms/step
Next char prediction : <stop>


In [2]:
from keras.models import Sequential
from keras.layers import SimpleRNN,Dense
import numpy as np

vocab = ['d','e','p','t','h']
char2idx = {c: i for i, c in enumerate(vocab)}
idx2char = {i: c for c, i in char2idx.items()}

seq = ['d','e','p','t']
target = ['e','p','t','h']

X = np.eye(len(vocab))[[char2idx[c] for c in seq]].reshape(len(seq), 1, len(vocab))
y = np.eye(len(vocab))[[char2idx[c] for c in target]]

model = Sequential([SimpleRNN(64, input_shape=(1, len(vocab))),
                   Dense(len(vocab), activation='softmax')
                   ])

model.compile(loss='categorical_crossentropy',optimizer='adam')

model.fit(X,y,epochs=200,verbose=0)

test = np.eye(len(vocab))[[char2idx['h']]].reshape(1,1,len(vocab))

pred = model.predict(test)
print('Next char prediction :', idx2char[np.argmax(pred)])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 355ms/step
Next char prediction : h


# OBSERVATIONS
1. we need a ending character for sure like . , <stop>
2. if we increase the number of perceptrons it can learn better

In [3]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Flatten
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

# 1. Prepare the data
reviews = ["good movie", "bad movie", "excellent film", "terrible acting"]
labels = np.array([1, 0, 1, 0]) # 1 for positive, 0 for negative

# Tokenize the text
tokenizer = Tokenizer(num_words=None) # Consider all words
tokenizer.fit_on_texts(reviews)
word_index = tokenizer.word_index
vocab_size = len(word_index) + 1 # +1 for the padding token

# Convert text to sequences of integers
sequences = tokenizer.texts_to_sequences(reviews)

# Pad sequences to have the same length
max_length = max(len(seq) for seq in sequences)
padded_sequences = pad_sequences(sequences, maxlen=max_length, padding='post')

# 2. Define the Keras model
embedding_dim = 10
model = Sequential([
    Embedding(vocab_size, embedding_dim, input_length=max_length), # Embed word indices into vectors
    SimpleRNN(8),                                              # Simple RNN layer with 8 hidden units
    Dense(1, activation='sigmoid')                             # Output layer with sigmoid for binary classification
])

# 3. Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# 4. Train the model (for a few epochs for demonstration)
model.fit(padded_sequences, labels, epochs=10, verbose=0)

# 5. Predict the sentiment of the first review
test_review = ["amazing movie"]
test_sequence = tokenizer.texts_to_sequences(test_review)
padded_test_sequence = pad_sequences(test_sequence, maxlen=max_length, padding='post')
prediction = model.predict(padded_test_sequence)
sentiment = "Positive" if prediction[0] > 0.5 else "Negative"

# 6. Print the result
print(f"Review: '{test_review[0]}', Predicted Sentiment: {sentiment} (Confidence: {prediction[0][0]:.2f})")


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 365ms/step
Review: 'amazing movie', Predicted Sentiment: Positive (Confidence: 0.50)


# #Predicting Next Word

In [4]:
from keras.models import Sequential
from keras.layers import SimpleRNN, Dense
import numpy as np

# Define a small vocabulary of words
vocab = ['deep', 'learning', 'is', '<stop>']
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}

# Input sequence of words
sequence = ['deep', 'learning', 'is']
target = ['learning', 'is', '<stop>']  # Next words

# One-hot encode words
X = np.eye(len(vocab))[[word2idx[w] for w in sequence]].reshape(len(sequence), 1, len(vocab))
y = np.eye(len(vocab))[[word2idx[w] for w in target]]

# Build a simple RNN model
model = Sequential([
    SimpleRNN(8, input_shape=(1, len(vocab))),
    Dense(len(vocab), activation='softmax')
])
model.compile(loss='categorical_crossentropy', optimizer='adam')

# Train the model
model.fit(X, y, epochs=300, verbose=0)

# Predict the next word after 'is'
test = np.eye(len(vocab))[word2idx['is']].reshape(1, 1, len(vocab))
pred = model.predict(test)
print("Next word prediction:", idx2word[np.argmax(pred)])


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 333ms/step
Next word prediction: <stop>


# LSTM

In [5]:
from keras.models import Sequential
from keras.layers import Embedding, LSTM, Dense
import numpy as np

# Vocabulary and word-index mapping
vocab = {'the': 0, 'sun': 1, 'is': 2, 'shining': 3, 'bright': 4, 'today': 5}
idx2word = {i: w for w, i in vocab.items()}
vocab_size = len(vocab)

# Input sequence: "the sun is shining"
X = np.array([[0, 1, 2, 3]])  # indices of words
y = np.array([4])  # target: "bright"

# Model
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=10, input_length=4),
    LSTM(32),  # LSTM can capture long-term dependencies
    Dense(vocab_size, activation='softmax')
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')

# Train
model.fit(X, y, epochs=200, verbose=0)

# Predict next word
test = np.array([[0, 1, 2, 3]])  # "the sun is shining"
pred = model.predict(test)
print("Predicted next word:", idx2word[np.argmax(pred)])
 

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 167ms/step
Predicted next word: bright


# Translation

In [6]:
from keras.models import Model
from keras.layers import Input, SimpleRNN, Dense
import numpy as np

# Dummy vocab (English + Telugu)
input_text = "how are you"
target_text = "మీరు ఎలా ఉన్నారు"
input_chars = sorted(set(input_text))
target_chars = sorted(set(target_text))

# Create token mappings
input_idx = {ch: i for i, ch in enumerate(input_chars)}
target_idx = {ch: i for i, ch in enumerate(target_chars)}
reverse_target_idx = {i: ch for ch, i in target_idx.items()}

# Encode input and target into one-hot vectors
def one_hot(text, length, char_set):
    data = np.zeros((1, length, len(char_set)))
    for t, ch in enumerate(text):
        data[0, t, char_set[ch]] = 1
    return data

enc_input = one_hot(input_text, len(input_text), input_idx)
dec_input = one_hot(target_text, len(target_text), target_idx)
dec_target = np.copy(dec_input)

# Build encoder-decoder model
enc_in = Input(shape=(None, len(input_idx)))
_, enc_state = SimpleRNN(64, return_state=True)(enc_in)

dec_in = Input(shape=(None, len(target_idx)))
dec_out, _ = SimpleRNN(64, return_sequences=True, return_state=True)(dec_in, initial_state=enc_state)
dec_out = Dense(len(target_idx), activation='softmax')(dec_out)

model = Model([enc_in, dec_in], dec_out)
model.compile(optimizer='adam', loss='categorical_crossentropy')
model.fit([enc_input, dec_input], dec_target, epochs=100, verbose=0)

# Inference: Predict translation
out = model.predict([enc_input, dec_input])
translated = ''.join([reverse_target_idx[np.argmax(vec)] for vec in out[0]])
print("Input:", input_text)
print("Predicted:", translated)


# New input for testing
test_input = "how"
enc_test = one_hot(test_input, len(test_input), input_idx)

# Reuse decoder input from training (or use a start token if available)
dec_test_input = one_hot(target_text, len(target_text), target_idx)

# Predict
output = model.predict([enc_test, dec_test_input])
translated = ''.join([reverse_target_idx[np.argmax(vec)] for vec in output[0]])
print("Input:", test_input)
print("Predicted Translation:", translated.strip())

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 519ms/step
Input: how are you
Predicted: మీరు ఎలా ఉన్నారు
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 531ms/step
Input: how
Predicted Translation: రీరు ఎలా ఉన్నారు
